# Golden Solution — MiniBench (Final)
Reference implementations for Subtasks A/B/C.

- Seasonal forcing amplitude increased to improve identifiability.
- Subtask C uses deterministic 2-stage grid search + Euler (dt=0.5 day) for tractable runtime.


In [ ]:

import math
import numpy as np

SIGMA = 5.670374419e-8

ALPHA_WARM = 0.30
ALPHA_COLD = 0.62
T0_TRUE = 273.15
K_TRUE  = 5.0

S0 = 1361.0

# Subtask A
S_A = S0
BRACKET_A = (210.0, 330.0)
TOL_A = 1e-8
MAX_ITER_A = 200

# Subtask B
C = 2.5e8
A_SEASON = 0.10
P_DAYS = 365.0
DT_DAYS = 0.25
N_DAYS = int(365.0 * 8.0)
T_INIT = 288.0

# Subtask C
OBS_SEED = 42
NOISE_STD = 0.3
OBS_EVERY_DAYS = 5.0
T0_BOUNDS = (265.0, 285.0)
K_BOUNDS  = (2.0, 10.0)
times_days = np.arange(365.0*6.0, 365.0*8.0, OBS_EVERY_DAYS)


In [ ]:

def _alpha_of_T(T: float, alpha_warm: float, alpha_cold: float, T0: float, k: float) -> float:
    if k <= 0:
        raise ValueError("k must be > 0")
    x = (T - T0) / k
    if x > 50:
        frac = 0.0
    elif x < -50:
        frac = 1.0
    else:
        frac = 1.0 / (1.0 + math.exp(x))
    alpha = alpha_warm + (alpha_cold - alpha_warm) * frac
    return max(0.0, min(1.0, alpha))


def _energy_balance(T: float, S: float, sigma: float, alpha_warm: float, alpha_cold: float, T0: float, k: float) -> float:
    alpha = _alpha_of_T(T, alpha_warm, alpha_cold, T0, k)
    return (1.0 - alpha) * (S / 4.0) - sigma * (T ** 4)


def equilibrium_temperature(S, sigma, alpha_warm, alpha_cold, T0, k, bracket=(150.0, 350.0), tol=1e-8, max_iter=200):
    a, b = float(bracket[0]), float(bracket[1])
    fa = _energy_balance(a, S, sigma, alpha_warm, alpha_cold, T0, k)
    fb = _energy_balance(b, S, sigma, alpha_warm, alpha_cold, T0, k)
    if fa == 0.0: return a
    if fb == 0.0: return b
    if fa * fb > 0:
        raise ValueError("Bracket does not contain a root (no sign change).")
    lo, hi = a, b
    flo = fa
    for _ in range(int(max_iter)):
        mid = 0.5*(lo+hi)
        fmid = _energy_balance(mid, S, sigma, alpha_warm, alpha_cold, T0, k)
        if abs(fmid) < tol or abs(hi-lo) < tol:
            return mid
        if flo * fmid <= 0:
            hi = mid
        else:
            lo = mid
            flo = fmid
    return 0.5*(lo+hi)


def simulate_temperature_series(C, S0, a, P_days, sigma, alpha_warm, alpha_cold, T0, k, T_init=288.0, dt_days=0.25, n_days=365*6):
    dt_sec = float(dt_days) * 86400.0
    P_sec = float(P_days) * 86400.0
    n_steps = int(round(float(n_days) / float(dt_days)))
    if n_steps <= 1:
        raise ValueError("n_days must be > dt_days.")

    def S_of_t(t_sec: float) -> float:
        return float(S0) * (1.0 + float(a) * math.cos(2.0 * math.pi * (t_sec / P_sec)))

    def dTdt(t_sec: float, T: float) -> float:
        S_t = S_of_t(t_sec)
        alpha = _alpha_of_T(T, alpha_warm, alpha_cold, T0, k)
        incoming = (1.0 - alpha) * (S_t / 4.0)
        outgoing = sigma * (T ** 4)
        return (incoming - outgoing) / float(C)

    t_days = np.zeros(n_steps + 1, dtype=float)
    T = np.zeros(n_steps + 1, dtype=float)
    T[0] = float(T_init)

    for i in range(n_steps):
        t0 = i * dt_sec
        Ti = T[i]
        # RK4
        k1 = dTdt(t0, Ti)
        k2 = dTdt(t0 + 0.5 * dt_sec, Ti + 0.5 * dt_sec * k1)
        k3 = dTdt(t0 + 0.5 * dt_sec, Ti + 0.5 * dt_sec * k2)
        k4 = dTdt(t0 + dt_sec, Ti + dt_sec * k3)
        T_next = Ti + (dt_sec / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        if not math.isfinite(T_next) or T_next <= 0:
            raise ValueError(f"Unphysical temperature at step {i}: {T_next}")
        T[i+1] = T_next
        t_days[i+1] = (i+1) * float(dt_days)
    return t_days, T


def simulate_temperature(C, S0, a, P_days, sigma, alpha_warm, alpha_cold, T0, k, T_init=288.0, dt_days=0.25, n_days=365*6):
    t_days, T = simulate_temperature_series(
        C=C, S0=S0, a=a, P_days=P_days, sigma=sigma,
        alpha_warm=alpha_warm, alpha_cold=alpha_cold, T0=T0, k=k,
        T_init=T_init, dt_days=dt_days, n_days=n_days
    )
    final_start = float(n_days) - 365.0
    T_final = T[t_days >= final_start]
    return {"T_mean": float(np.mean(T_final)), "T_amp": float(0.5*(np.max(T_final)-np.min(T_final)))}


def calibrate_T0_k(observations, times_days, C, S0, a, P_days, sigma, alpha_warm, alpha_cold,
                   T_init=288.0, dt_days=0.25, search_bounds=((250.0, 290.0), (2.0, 20.0))):
    """Deterministic 2-stage grid search using Euler dt=0.5 day."""
    obs = np.asarray(observations, dtype=float)
    t_obs = np.asarray(times_days, dtype=float)
    if obs.shape != t_obs.shape:
        raise ValueError("observations and times_days must have the same shape.")

    (T0_lo, T0_hi), (k_lo, k_hi) = search_bounds
    dt_cal = 0.5
    dt_sec = dt_cal * 86400.0
    P_sec = float(P_days) * 86400.0
    n_days_needed = float(np.max(t_obs) + 10.0)
    n_steps = int(math.ceil(n_days_needed / dt_cal))

    # Precompute times for integration
    t_series = np.arange(n_steps + 1, dtype=float) * dt_cal  # days

    def S_of_t_sec(t_sec: float) -> float:
        return float(S0) * (1.0 + float(a) * math.cos(2.0 * math.pi * (t_sec / P_sec)))

    def simulate_candidate(T0, k):
        T = float(T_init)
        T_out = np.zeros(n_steps + 1, dtype=float)
        T_out[0] = T
        for i in range(n_steps):
            t_sec = (i * dt_cal) * 86400.0
            S_t = S_of_t_sec(t_sec)
            alpha = _alpha_of_T(T, alpha_warm, alpha_cold, float(T0), float(k))
            incoming = (1.0 - alpha) * (S_t / 4.0)
            outgoing = sigma * (T ** 4)
            dTdt = (incoming - outgoing) / float(C)
            T = T + dTdt * dt_sec
            if not math.isfinite(T) or T <= 0:
                return None
            T_out[i+1] = T
        return T_out

    def sse_for(T0, k):
        T_out = simulate_candidate(T0, k)
        if T_out is None:
            return float("inf")
        pred = np.interp(t_obs, t_series, T_out)
        err = pred - obs
        return float(np.sum(err*err))

    # Stage 1 coarse
    T0_grid_1 = np.linspace(T0_lo, T0_hi, 13)
    k_grid_1  = np.linspace(k_lo,  k_hi,  13)
    best = (None, None, float("inf"))
    for T0 in T0_grid_1:
        for k in k_grid_1:
            v = sse_for(T0, k)
            if v < best[2]:
                best = (float(T0), float(k), float(v))

    # Stage 2 refine around best
    T0_c, k_c, _ = best
    T0_grid_2 = np.linspace(max(T0_lo, T0_c-2.0), min(T0_hi, T0_c+2.0), 21)
    k_grid_2  = np.linspace(max(k_lo,  k_c-2.0),  min(k_hi,  k_c+2.0),  21)
    best2 = (T0_c, k_c, best[2])
    for T0 in T0_grid_2:
        for k in k_grid_2:
            v = sse_for(T0, k)
            if v < best2[2]:
                best2 = (float(T0), float(k), float(v))

    return {"T0_hat": best2[0], "k_hat": best2[1], "sse": best2[2]}


In [ ]:

rng = np.random.default_rng(OBS_SEED)
t_dense, T_dense = simulate_temperature_series(
    C=C, S0=S0, a=A_SEASON, P_days=P_DAYS, sigma=SIGMA,
    alpha_warm=ALPHA_WARM, alpha_cold=ALPHA_COLD,
    T0=T0_TRUE, k=K_TRUE,
    T_init=T_INIT, dt_days=DT_DAYS, n_days=N_DAYS
)
T_clean = np.interp(times_days, t_dense, T_dense)
observations = T_clean + rng.normal(0.0, NOISE_STD, size=T_clean.shape)

T_star_ref = equilibrium_temperature(S_A, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, BRACKET_A, TOL_A, MAX_ITER_A)
metrics_ref = simulate_temperature(C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, T_INIT, DT_DAYS, N_DAYS)
calib_ref = calibrate_T0_k(observations, times_days, C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T_INIT, DT_DAYS, (T0_BOUNDS, K_BOUNDS))

print("Ref A:", T_star_ref)
print("Ref B:", metrics_ref)
print("Ref C:", calib_ref)


Ref A: 218.5214850353077
Ref B: {'T_mean': 224.37736057853252, 'T_amp': 0.9115755622288901}
Ref C: {'T0_hat': 272.93333333333334, 'k_hat': 4.133333333333333, 'sse': 9.685780654854366}


In [ ]:

import unittest

class TestGolden(unittest.TestCase):
    def test_A_residual(self):
        T = equilibrium_temperature(S_A, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, BRACKET_A, TOL_A, MAX_ITER_A)
        self.assertLess(abs(_energy_balance(T, S_A, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE)), 1e-6)

    def test_B_keys(self):
        out = simulate_temperature(C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T0_TRUE, K_TRUE, T_INIT, DT_DAYS, N_DAYS)
        self.assertIn("T_mean", out)
        self.assertIn("T_amp", out)

    def test_C_accuracy(self):
        out = calibrate_T0_k(observations, times_days, C, S0, A_SEASON, P_DAYS, SIGMA, ALPHA_WARM, ALPHA_COLD, T_INIT, DT_DAYS, (T0_BOUNDS, K_BOUNDS))
        self.assertLess(abs(out["T0_hat"] - T0_TRUE), 2.0)
        self.assertLess(abs(out["k_hat"] - K_TRUE), 2.0)

unittest.main(argv=["-v"], exit=False)


...
----------------------------------------------------------------------
Ran 3 tests in 5.175s

OK
